In [2]:
# ! pip install langchain langchain-openai

In [3]:
pip show langchain

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/kanavbansal/Developer/github/bansalkanav/Generative-AI-Scratch-2-Advance-By-ThatAIGuy/07. Agentic Systems in LangChain/6. Model Context Protocol/3. Building MCP Server/.env/lib/python3.13/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
                 model="gpt-4o-mini",
                 temperature=0.0)

In [7]:
# ! pip install langchain-mcp-adapters

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [9]:
client = MultiServerMCPClient(
    {
        "mathematical-operations": {
          "transport": "stdio",
          "command": "python",
          "args": [
            "mcp-server.py"
          ]
        }
    }
)

In [10]:
time_toolset = await client.get_tools()

time_toolset

[StructuredTool(name='multiply', description='This tool takes two integers and returns the product', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x11190dbc0>)]

In [11]:
print(f"Loaded {len(time_toolset)} MCP Tools: {[tool.name for tool in time_toolset]}")

Loaded 1 MCP Tools: ['multiply']


In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=time_toolset
)

In [13]:
response = await agent.ainvoke({"messages": "What time is it?"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What time is it?
================================== Ai Message ==================================

I don't have real-time capabilities to check the current time. You can easily find the time by checking your device's clock or using a search engine.


In [14]:
response = await agent.ainvoke({"messages": "What is 123456 multiplied with 789012?"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is 123456 multiplied with 789012?
================================== Ai Message ==================================
Tool Calls:
  multiply (call_SG3OyMfNMg8RQlgRZQYvg88M)
 Call ID: call_SG3OyMfNMg8RQlgRZQYvg88M
  Args:
    a: 123456
    b: 789012
================================= Tool Message =================================
Name: multiply

[{'type': 'text', 'text': '97408265472', 'id': 'lc_5f467c7b-5289-42b7-804e-a2ed156f36c4'}]
================================== Ai Message ==================================

123456 multiplied by 789012 equals 97,408,265,472.
